# Zipformer 30M — Colab T4

**Model:** [hynt/Zipformer-30M-RNNT-6000h](https://huggingface.co/hynt/Zipformer-30M-RNNT-6000h)

Dùng checkpoint **ONNX FP32**, yêu cầu CUDA provider. Không âm thầm đổi qua CPU. Dành cho thử nghiệm thực tế; chưa kiểm thử trực tiếp trên Colab T4 trong môi trường tạo notebook.

## 1. Kiểm tra Colab T4
Runtime → Change runtime type → T4 GPU. Cell dưới sẽ dừng nếu không có GPU.

In [1]:
import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Chưa có CUDA: hãy chọn T4 GPU và kết nối lại runtime"
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 2. Cài đặt
Sử dụng checkpoint **FP32** gốc của `hynt/Zipformer-30M-RNNT-6000h`, không dùng INT8 vì CUDA có thể fallback. `sherpa-onnx` được yêu cầu phải thực sự có CUDA provider. Nếu wheel không hỗ trợ GPU, cell kiểm tra sẽ dừng thay vì âm thầm chạy CPU.

In [2]:
%pip install -q \
    "huggingface_hub>=0.25,<1.0" \
    soundfile \
    sentencepiece \
    sherpa-onnx \
    onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.18.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Tải 3 ONNX FP32 và tokens từ model gốc

In [3]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_ID = "hynt/Zipformer-30M-RNNT-6000h"

model_dir = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=[
            "encoder-epoch-20-avg-10.onnx",
            "decoder-epoch-20-avg-10.onnx",
            "joiner-epoch-20-avg-10.onnx",
            "bpe.model",
        ],
    )
)

print("Model directory:", model_dir)

for name in [
    "encoder-epoch-20-avg-10.onnx",
    "decoder-epoch-20-avg-10.onnx",
    "joiner-epoch-20-avg-10.onnx",
    "bpe.model",
]:
    assert (model_dir / name).exists(), name
    print("OK:", name)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

bpe.model:   0%|          | 0.00/268k [00:00<?, ?B/s]

Model directory: /root/.cache/huggingface/hub/models--hynt--Zipformer-30M-RNNT-6000h/snapshots/ad6f873b5f5b6ed083821253a88d81afed2505c4
OK: encoder-epoch-20-avg-10.onnx
OK: decoder-epoch-20-avg-10.onnx
OK: joiner-epoch-20-avg-10.onnx
OK: bpe.model


In [4]:
import urllib.request
import tarfile
from pathlib import Path

url = (
    "https://github.com/k2-fsa/sherpa-onnx/releases/"
    "download/asr-models/"
    "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09.tar.bz2"
)

archive = Path("/content/zipformer_tokens.tar.bz2")

if not archive.exists():
    print("Downloading official sherpa-onnx package...")
    urllib.request.urlretrieve(url, archive)

with tarfile.open(archive, "r:bz2") as tar:
    token_files = [
        member for member in tar.getmembers()
        if Path(member.name).name == "tokens.txt"
        and member.isfile()
    ]

    if len(token_files) != 1:
        raise RuntimeError("Không tìm thấy duy nhất một tokens.txt")

    source = tar.extractfile(token_files[0])
    if source is None:
        raise RuntimeError("Không đọc được tokens.txt")

    token_data = source.read()

tokens_path = Path("/content/zipformer_tokens.txt")
tokens_path.write_bytes(token_data)

print("Tokens file:", tokens_path)
print("Size:", tokens_path.stat().st_size, "bytes")

Tokens file: /content/zipformer_tokens.txt
Size: 23238 bytes


## 2. Upload audio
Cùng một file WAV/MP3 ở cả hai notebook để so sánh. Chuyển về mono PCM 16 kHz.

In [5]:
from google.colab import files
from pathlib import Path
import subprocess, soundfile as sf
uploaded = files.upload()
assert uploaded, "Hãy upload ít nhất một file audio"
original = next(iter(uploaded))
audio_path = '/content/asr_test_16k.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',original,'-ac','1','-ar','16000','-c:a','pcm_s16le',audio_path],check=True)
signal,sr = sf.read(audio_path)
duration = len(signal)/sr
print(f'Audio: {audio_path} | {duration:.2f}s | {sr}Hz')

Saving p_5694761_156.mp3 to p_5694761_156.mp3
Audio: /content/asr_test_16k.wav | 10.35s | 16000Hz


## 4. Kiểm tra CUDA ở ONNX Runtime và Sherpa
Nếu gặp lỗi CUDA provider không khả dụng, runtime `sherpa-onnx` của phiên bản pip chưa hỗ trợ CUDA. **Không đổi thành CPU rồi báo là GPU.**

In [6]:
import onnxruntime as ort
print('ONNX Runtime providers:',ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'onnxruntime-gpu chưa nhận CUDA; xem compatibility CUDA/cuDNN'
import sherpa_onnx
print('sherpa-onnx loaded')

ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
sherpa-onnx loaded


## 5. Tạo recognizer GPU (chỉ tạo một lần)
Giữ recognizer trong notebook để benchmark **warm inference**, không chạy subprocess cho mỗi audio.

In [7]:
import time
import sherpa_onnx
start_load=time.perf_counter()
recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    tokens=str(tokens_path),
    encoder=str(model_dir/'encoder-epoch-20-avg-10.onnx'),
    decoder=str(model_dir/'decoder-epoch-20-avg-10.onnx'),
    joiner=str(model_dir/'joiner-epoch-20-avg-10.onnx'),
    num_threads=1,
    sample_rate=16000,
    feature_dim=80,
    decoding_method='greedy_search',
    provider='cuda',
)
load_time=time.perf_counter()-start_load
print('Recognizer initialized with provider=cuda; load time:',round(load_time,3),'s')

Recognizer initialized with provider=cuda; load time: 0.983 s


## 6. Upload file thử và chạy benchmark
Cell upload nằm ở đầu notebook để tiện tái sử dụng. Warm-up một lần, sau đó đo nhiều lần trong cùng Python process.

In [9]:
import numpy as np, soundfile as sf, time, statistics, json
from pathlib import Path
signal, sr=sf.read(audio_path, dtype='float32')
assert sr==16000 and signal.ndim==1

def recognize_once():
    t0=time.perf_counter()
    stream=recognizer.create_stream()
    stream.accept_waveform(sr, signal)
    recognizer.decode_stream(stream)
    dt=time.perf_counter()-t0
    return stream.result.text, dt

recognize_once()  # warmup
results=[recognize_once() for _ in range(5)]
latencies=[t for _,t in results]
median=statistics.median(latencies)
print('TRANSCRIPT:', results[-1][0])
print(f'Audio: {duration:.3f}s | warm request median: {median:.3f}s | RTF={median/duration:.4f}')
print('Individual request times:',[round(t,4) for t in latencies])
report={'model':'hynt/Zipformer-30M-RNNT-6000h','runtime':'sherpa-onnx ONNX FP32','requested_provider':'cuda',
 'audio_seconds':duration,'warm_request_median_seconds':median,'rtf':median/duration,'text':results[-1][0]}
Path('/content/zipformer30m_results.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))

TRANSCRIPT: VĂN BẢN NÀY CÓ THỂ ĐƯỢC THAY THẾ BẰNG BẤT KỲ VĂN BẢN NÀO TRONG HỘP VĂN BẢN NÀY BẠN CÓ THỂ NHẬP HOẶC DÁN VĂN BẢN CỦA RIÊNG MÌNH BẠN CÓ THỂ SỬ DỤNG NHIỀU GIỌNG NÓI CHO MỘT VĂN BẢN
Audio: 10.345s | warm request median: 0.397s | RTF=0.0384
Individual request times: [0.3923, 0.3962, 0.4107, 0.4086, 0.397]


417

**Benchmark:** đo `create_stream` + `accept_waveform` + `decode_stream` trong cùng process, không tính load model và không tính đọc file. GPU được yêu cầu qua `provider="cuda"`; tùy bản build, hãy kiểm tra provider thật khi triển khai. Nếu lỗi không hỗ trợ CUDA, cần dùng bản sherpa-onnx có GPU support.